In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

# Load app IDs and take the first 50
with open("cleaned_steam_apps.json", "r", encoding="utf-8") as f:
    app_data = json.load(f)
app_ids = [app["appid"] for app in app_data["applist"]["apps"]][50001:]

# Set up a session with retries
session = requests.Session()
retry_strategy = Retry(
    total=3,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504],
)
adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

def fetch_app_data(app_id):
    try:
        response = session.get(f"https://store.steampowered.com/app/{app_id}/", headers=headers, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "html.parser")

        # Name
        name_elem = soup.find("div", {"id": "appHubAppName"})
        name = name_elem.text.strip() if name_elem else None

        # Tags (first 3)
        tags = [t.string.strip() for t in soup.find_all("a", {"class": "app_tag"})][:3] if name else []

        # Popular Tags (all)
        popular_tags_div = soup.find("div", {"class": "glance_tags popular_tags"})
        popular_tags = [t.string.strip() for t in popular_tags_div.find_all("a", {"class": "app_tag"})] if popular_tags_div else []

        # User Reviews
        reviews_section = soup.find("div", {"id": "userReviews"})
        all_reviews = None
        all_percent = None
        all_count = None

        if reviews_section:

            all_row = reviews_section.find("div", {"class": "user_reviews_summary_row", "data-tooltip-html": lambda x: x and "last 30 days" not in x})
            if all_row:
                all_reviews = all_row.find("span", {"class": "game_review_summary"}).text.strip() if all_row.find("span", {"class": "game_review_summary"}) else None
                all_tooltip = all_row.get("data-tooltip-html", "")
                if all_tooltip:
                    parts = all_tooltip.split(" of the ")
                    all_percent = parts[0].strip() if parts else None
                    all_count = parts[1].split(" user")[0].replace(",", "") if len(parts) > 1 else None

        # Release Date
        release_elem = soup.find("div", {"class": "date"})
        release_date = release_elem.text.strip() if release_elem else None

        # Developer
        dev_elem = soup.select_one(".details_block b:contains('Developer:')")
        developer = dev_elem.find_next("a").text.strip() if dev_elem else None

        # Publisher
        pub_elem = soup.select_one(".details_block b:contains('Publisher:')")
        publisher = pub_elem.find_next("a").text.strip() if pub_elem else None

        # Price
        price_elem = soup.find("div", {"class": "game_purchase_price"})
        price = price_elem.text.strip() if price_elem else "Free to Play" if soup.find(text="Free to Play") else None

        # Description
        desc_elem = soup.find("div", {"class": "game_description_snippet"})
        description = desc_elem.text.strip() if desc_elem else None

        # Store result
        result = {
            "app_id": app_id,
            "name": name,
            "tags": ",".join(tags),
            "popular_tags": ",".join(popular_tags),
            "all_reviews": all_reviews,
            "all_percent": all_percent,
            "all_count": all_count,
            "release_date": release_date,
            "developer": developer,
            "publisher": publisher,
            "price": price,
            "description": description
        }
        print(f"Success: {app_id} - {name}")
        return result

    except requests.exceptions.Timeout as e:
        print(f"Timeout after retries: {app_id} - {e}")
        return {"app_id": app_id, "name": None, "tags": None, "popular_tags": None, "recent_reviews": None, "recent_percent": None, "recent_count": None, "all_reviews": None, "all_percent": None, "all_count": None, "release_date": None, "developer": None, "publisher": None, "price": None, "description": None}
    except requests.exceptions.RequestException as e:
        print(f"Other error: {app_id} - {e}")
        return {"app_id": app_id, "name": None, "tags": None, "popular_tags": None, "recent_reviews": None, "recent_percent": None, "recent_count": None, "all_reviews": None, "all_percent": None, "all_count": None, "release_date": None, "developer": None, "publisher": None, "price": None, "description": None}

# Use ThreadPoolExecutor with max_workers
data = []
max_workers = 3
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    future_to_app = {executor.submit(fetch_app_data, app_id): app_id for app_id in app_ids}
    for future in as_completed(future_to_app):
        app_id = future_to_app[future]
        try:
            result = future.result()
            data.append(result)
        except Exception as e:
            print(f"Exception for app ID {app_id}: {e}")

# Save to CSV with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"steam_game_descriptions_test_{timestamp}.csv"
df = pd.DataFrame(data)
df.to_csv(filename, index=False)
print(f"Saved {len(data)} games to {filename}")

<ipython-input-1-953ecc014297>:75: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  price = price_elem.text.strip() if price_elem else "Free to Play" if soup.find(text="Free to Play") else None


Streaming output truncated to the last 5000 lines.
Success: 3346840 - Rembrunir - Onnu
Success: 3346900 - None
Success: 3346940 - Dough Slapping: A Touch to Desire
Success: 3346970 - Anatta
Success: 3347020 - Punch Party
Success: 3347060 - 101 Cats Hidden in Miami
Success: 3347070 - 101 Cats Hidden in Rome
Success: 3347050 - 101 Cats Hidden in Singapore
Success: 3347080 - None
Success: 3347150 - Crimson Harvest
Success: 3347120 - Above the Cloud: Time Bound Mansion
Success: 3347210 - Stage Break Idle
Success: 3347200 - Cat Gladiator
Success: 3347290 - Spellslime
Success: 3347280 - .HEADSPACE
Success: 3347310 - Let's Build Wonders: Galaxy
Success: 3347330 - Realm Traveler
Success: 3347400 - GIRLS' FRONTLINE 2: EXILIUM
Success: 3347420 - :3
Success: 3347370 - Morrigan's Isle
Success: 3347430 - 小石游记 Little Stone Journey
Success: 3347520 - Mining
Success: 3347500 - Meow Island
Success: 3347560 - Binary Game
Success: 3347670 - None
Success: 3347650 - SOUNDSCAPE
Success: 3347710 - Baoli
Succ

In [ ]:
import requests
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

# Load and combine CSV files into one array
data_1 = pd.read_csv("steam_game_descriptions_test_20250304_125622_dropAllNull.csv")
data_2 = pd.read_csv("steam_game_descriptions_test_20250304_222258_dropAllNull.csv")
data = pd.concat([data_1, data_2])
app_ids = data["app_id"][15001:].tolist()  # All app IDs in one array
print(f"Total app IDs to process: {len(app_ids)}")

# Set up a session with retries
session = requests.Session()
retry_strategy = Retry(
    total=3,
    backoff_factor=2,
    status_forcelist=[429, 500, 502, 503, 504],
)
adapter = HTTPAdapter(max_retries=retry_strategy, pool_maxsize=20)
session.mount("https://", adapter)

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

def fetch_reviews(app_id):
    reviews = []
    try:
        url = f"https://store.steampowered.com/appreviews/{app_id}?json=1&num_per_page=100"
        response = session.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        data = response.json()

        if data["success"] != 1:
            print(f"No reviews available for app ID {app_id}")
            return [{"app_id": app_id, "recommendationid": None, "review": None, "voted_up": None,
                     "timestamp_created": None, "playtime_forever": None,
                     "weighted_vote_score": None, "votes_up": None, "steamid": None}]

        for review in data["reviews"]:
            reviews.append({
                "app_id": app_id,
                "recommendationid": review.get("recommendationid"),
                "review": review.get("review"),
                "voted_up": review.get("voted_up"),
                "timestamp_created": review.get("timestamp_created"),
                "playtime_forever": review["author"].get("playtime_forever"),
                "weighted_vote_score": float(review.get("weighted_vote_score", 0)),
                "votes_up": review.get("votes_up", 0),
                "steamid": review["author"].get("steamid")
            })
        print(f"Success: Fetched {len(reviews)} reviews for app ID {app_id}")
        return reviews

    except requests.exceptions.Timeout as e:
        print(f"Timeout after retries: {app_id} - {e}")
        return [{"app_id": app_id, "recommendationid": None, "review": None, "voted_up": None,
                 "timestamp_created": None, "playtime_forever": None,
                 "weighted_vote_score": None, "votes_up": None, "steamid": None}]
    except requests.exceptions.RequestException as e:
        print(f"Other error: {app_id} - {e}")
        return [{"app_id": app_id, "recommendationid": None, "review": None, "voted_up": None,
                 "timestamp_created": None, "playtime_forever": None,
                 "weighted_vote_score": None, "votes_up": None, "steamid": None}]

# Use ThreadPoolExecutor with max_workers
data = []
max_workers = 3
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    future_to_app = {executor.submit(fetch_reviews, app_id): app_id for app_id in app_ids}
    for future in as_completed(future_to_app):
        app_id = future_to_app[future]
        try:
            results = future.result()
            data.extend(results)
        except Exception as e:
            print(f"Exception for app ID {app_id}: {e}")

# Save to CSV with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"steam_game_reviews_all_{timestamp}.csv"
df = pd.DataFrame(data)
df.to_csv(filename, index=False)
print(f"Saved {len(data)} reviews to {filename}")

Streaming output truncated to the last 5000 lines.
Success: Fetched 1 reviews for app ID 3137350
Success: Fetched 14 reviews for app ID 3137740
Success: Fetched 1 reviews for app ID 3137910
Success: Fetched 0 reviews for app ID 3137780
Success: Fetched 1 reviews for app ID 3137930
Success: Fetched 1 reviews for app ID 3138140
Success: Fetched 0 reviews for app ID 3138020
Success: Fetched 0 reviews for app ID 3138100
Success: Fetched 1 reviews for app ID 3138220
Success: Fetched 0 reviews for app ID 3138210
Success: Fetched 0 reviews for app ID 3138270
Success: Fetched 2 reviews for app ID 3138280
Success: Fetched 1 reviews for app ID 3138360
Success: Fetched 0 reviews for app ID 3138380
Success: Fetched 2 reviews for app ID 3138570
Success: Fetched 1 reviews for app ID 3138640
Success: Fetched 2 reviews for app ID 3138510
Success: Fetched 1 reviews for app ID 3138660
Success: Fetched 0 reviews for app ID 3138680
Success: Fetched 2 reviews for app ID 3138690
Success: Fetched 1 reviews f

Streaming output truncated to the last 5000 lines.
Success: Fetched 19 reviews for app ID 1437030
Success: Fetched 2 reviews for app ID 1437040
Success: Fetched 3 reviews for app ID 1437060
Success: Fetched 1 reviews for app ID 1437100
Success: Fetched 7 reviews for app ID 1437120
Success: Fetched 2 reviews for app ID 1437160
Success: Fetched 3 reviews for app ID 1437170
Success: Fetched 0 reviews for app ID 1437210
Success: Fetched 8 reviews for app ID 1437270
Success: Fetched 1 reviews for app ID 1437340
Success: Fetched 5 reviews for app ID 1437370
Success: Fetched 33 reviews for app ID 1437400
Success: Fetched 5 reviews for app ID 1437410
Success: Fetched 10 reviews for app ID 1437420
Success: Fetched 4 reviews for app ID 1437520
Success: Fetched 1 reviews for app ID 1437480
Success: Fetched 3 reviews for app ID 1437580
Success: Fetched 0 reviews for app ID 1437600
Success: Fetched 0 reviews for app ID 1437680
Success: Fetched 1 reviews for app ID 1437750
Success: Fetched 1 reviews